<a href="https://colab.research.google.com/github/jonik2909/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch bitsandbytes datasets peft

In [2]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

In [3]:
# 1. Configuration and Tokenizer
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# 2. Quantization Setup (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

# 3. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # Automatically handles GPU/CPU placement
    # dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
# 4. Inference Test (Before Fine-tuning)
# prompt = "Explain what a tokenizer is."
prompt = "A tokenizer is a tool in natural language processing that"

# Prepare inputs and move to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

# Decode and print results
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# first_block = model.model.layers[0]
# print("first_block:", first_block)
# print(first_block.self_attn)
# print(model.config)

A tokenizer is a tool in natural language processing that splits a sentence into its constituent words. The basic steps involved in tokenization involve splitting each word into its constituent parts (or parts of speech) based on its part-of-speech (POS) tag.

Step 1: Tokenization (Part-of-Speech Tagging)

The first step in tokenization is to determine the part-of-


In [5]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including fronzen 4-bit): {total_params:,}")

Total parameters (including fronzen 4-bit): 615,606,272


## datasets library | load_dataset
* intruction tuning

In [6]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset[0]

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [ ]:
def generate_prompt(example):
  intsruction = example['instruction']
  input_text = example['input']
  output_Text = example['output']

  if input_text:
    return (
        "### Instruction:\n"
        f"{intsruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_Text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{intsruction}\n\n"
        "### Response:\n"
        f"{output_Text}\n\n"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset[0]['text']

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
dataset

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

In [ ]:
# Define the LoRA Configuration
lora_config = LoraConfig(
    r=8,                # Rank: lower numbers mean fewer parameters
    lora_alpha=16,      # Scaling factor for the learned weights
    lora_dropout=0.05,  # Dropout probability to prevent overfitting
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"], # Targets the Attention Query and Value matrices
)

In [ ]:
# Wrap the base model with LoRA layers
model = get_peft_model(model, lora_config)

In [ ]:
# Verify the reduction in trainable parameters
model.print_trainable_parameters()